# 06 · Mid-term reward oracle

The full project used a large configuration-wise reward oracle. For the mid-term we simplify it to **five wavelet actions**.

For each action:

1. train a logistic-regression classifier using only that wavelet feature group;
2. obtain patient-grouped out-of-fold probabilities on the training split;
3. fit the classifier on all training patients and obtain validation/test probabilities;
4. define a simple per-recording reward from the probability assigned to the true class.

Reward:

**r = 2 × p(correct class) − 1**

This gives a continuous reward in [-1, 1]:

- +1 = confident correct prediction
- 0 = uncertain
- -1 = confident incorrect prediction

This is intentionally simpler than the final weighted clinical reward.


In [1]:
%run ../common/00_config.ipynb
use_half("midsem", oracle="classical")

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
)

bundle = joblib.load(P["processed"] / "midterm_wavelet_features.joblib")
feats = bundle["features"]
wavelets = bundle["wavelets"]
feature_keys = bundle["feature_keys"]

df = pd.read_csv(P["metadata"] / "records_qc.csv")
df = df.set_index("record_id").loc[feats["record_id"]].reset_index()

labelled = df.murmur_label.isin(["Present", "Absent"])
df = df[labelled].reset_index(drop=True)

# Re-index feature arrays to the labelled recordings.
record_ids = df.record_id.tolist()
original_ids = feats["record_id"]
idx = np.array([original_ids.index(r) for r in record_ids])

X_by_action = {
    w: feats["clean"][k][idx]
    for w, k in zip(wavelets, feature_keys)
}

y = (df.murmur_label == "Present").astype(int).to_numpy()
groups = df.patient_id.to_numpy()
split = df.split.to_numpy()

tr = np.where(split == "train")[0]
va = np.where(split == "val")[0]
te = np.where(split == "test")[0]

print("Actions:", wavelets)
print("Train:", len(tr), "Val:", len(va), "Test:", len(te))


Python environment is working
PyTorch: 2.14.0+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
CirCor version 1.0.3
midsem: results -> results/midsem/ | reward oracle: classical


Actions: ['db4', 'db6', 'db8', 'sym4', 'coif3']
Train: 2100 Val: 450 Test: 457


## Patient-wise out-of-fold reward generation

In [2]:
oof_prob = np.zeros((len(df), len(wavelets)), dtype=np.float32)
val_prob = np.zeros((len(df), len(wavelets)), dtype=np.float32)
test_prob = np.zeros((len(df), len(wavelets)), dtype=np.float32)

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

for a, wv in enumerate(wavelets):
    X = X_by_action[wv]

    # OOF predictions only for the training split.
    for fold_tr, fold_va in cv.split(X[tr], y[tr], groups[tr]):
        fit_idx = tr[fold_tr]
        hold_idx = tr[fold_va]

        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
        )
        model.fit(X[fit_idx], y[fit_idx])
        oof_prob[hold_idx, a] = model.predict_proba(X[hold_idx])[:, 1]

    # Final model for validation/test.
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
    )
    model.fit(X[tr], y[tr])

    val_prob[va, a] = model.predict_proba(X[va])[:, 1]
    test_prob[te, a] = model.predict_proba(X[te])[:, 1]

def true_class_probability(prob, y_true):
    return np.where(y_true == 1, prob, 1.0 - prob)

reward_train = np.zeros((len(tr), len(wavelets)), dtype=np.float32)
for a in range(len(wavelets)):
    p = true_class_probability(oof_prob[tr, a], y[tr])
    reward_train[:, a] = 2.0 * p - 1.0

joblib.dump(
    {
        "wavelets": wavelets,
        "record_ids": record_ids,
        "y": y,
        "groups": groups,
        "split": split,
        "train_idx": tr,
        "val_idx": va,
        "test_idx": te,
        "reward_train": reward_train,
        "oof_prob": oof_prob,
        "val_prob": val_prob,
        "test_prob": test_prob,
    },
    P["processed"] / "midterm_reward_data.joblib",
    compress=3
)

print("Reward matrix shape:", reward_train.shape)
print("Mean reward by action:")
for a, wv in enumerate(wavelets):
    print(f"  {wv}: {reward_train[:, a].mean():.4f}")


Reward matrix shape: (2100, 5)
Mean reward by action:
  db4: 0.1517
  db6: 0.1010
  db8: 0.2951
  sym4: 0.2876
  coif3: 0.2874


## Validation comparison of the five fixed wavelet policies

In [3]:
def metrics_from_prob(y_true, p):
    pred = (p >= 0.5).astype(int)
    return {
        "macro_f1": f1_score(y_true, pred, average="macro"),
        "balanced_acc": balanced_accuracy_score(y_true, pred),
        "auroc": roc_auc_score(y_true, p)
    }

rows = []
for a, wv in enumerate(wavelets):
    m = metrics_from_prob(y[va], val_prob[va, a])
    rows.append({"wavelet": wv, **m})

wavelet_results = pd.DataFrame(rows)
wavelet_results.to_csv(P["tables"] / "06_midterm_wavelet_results.csv", index=False)
wavelet_results


,wavelet,macro_f1,balanced_acc,auroc
0,db4,0.593399,0.650913,0.718265
1,db6,0.489229,0.545213,0.592905
2,db8,0.591269,0.619357,0.674957
3,sym4,0.609692,0.656016,0.688686
4,coif3,0.585095,0.620903,0.662342
